### Get clear Categories from data after cleaning (Such as checking for language)

In [1]:
import re
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import torch
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from transformers import DataCollatorWithPadding
from sklearn.preprocessing import LabelEncoder
from datasets import Dataset
from sklearn.utils import resample
from collections import Counter
from tqdm import tqdm



In [ ]:
df = pd.read_pickle("../data/working/dedup_preprocessed_rev2_docs_since_2020_01_01_only_de_strict_only_long_name.pkl.gz")
df = df[["id", "product_id", "name", "desc", "brand", "shop_cat", "price"]]
print("Amount of distinct shop categories:", df["shop_cat"].nunique())

Amount of distinct shop categories: 161947


In [3]:

# ------------------------------------------------------------
# 0) VORAUSSETZUNGEN
# ------------------------------------------------------------

TOP_CATEGORIES = [
    "Auto & Motorrad",
    "Möbel & Wohnen",
    "Werkzeug & Baumarkt",
    "Elektronik & Computer",
    "Spielzeug & Baby",
    "Kleidung & Accessoires",
    "Kosmetik & Drogerie",
    "Lebensmittel & Getränke",
    "Gesundheit & Pflege",
    "Bücher, Filme & Musik",
    "Sport & Freizeit",
    "Haustier & Tierbedarf",
    "Bürobedarf"
]
# Heuristische Keyword-Regeln mit direkter, eindeutiger Zuordnung
# (falls ein Wort aus der Liste vorkommt → sofortige Zuordnung)
HEURISTIC_RULES = {
    "Auto & Motorrad": [
        "pkw", "motorradhelm", "neureifen", "autoreifen",
        "sommerreifen", "winterreifen", "allwetterreifen",
        "felge", "reifen", "kfz", "auto", "scheibenwischer",
        "motor", "motorrad", "scooter", "moped", "autoteil",
        "kennzeichen", "dachbox", "dachträger", "wagenheber", "motoröl",
    ],
    "Elektronik & Computer": [
        "gardine", "haushaltsgerät", "technik", "telefon","laptop", "smartphone", "fernseher", "tv", 
        "monitor", "kamera", "router", "konsole", "kühlschrank", "herd", "ofen", "computer", "elektronik", 
        "pc", "handy", "notebook", "handy", "tablet", "smartwatch", "bildschirm", "objektiv", "kopfhörer", 
        "router", "netzwerk", "tastatur", "konsole", "playstation", "xbox", "nintendo", "grafikkarte", "multimedia",
        "trockner", "waschmaschine", "spülmaschine", "küchenkleingerät", "küchengerät", "elektrogerät", 
        "digitalcamera", "digitalkamera"
    ],
    "Möbel & Wohnen": [
        "matratze", "bett", "sofa", "kommode", "lampe", "beleuchtung", "vorhang", "kissen", "teppich",  "wohnzimmer", "schlafzimmer", "zimmer", "wohnen", "möbel", "bad", "tisch", "stuhl", "regal", "schrank", "couch",
        "möbelzubehör", "pfanne", "geschirr", "topf", "töpfe", "esszimmer", "grill", "bettwäsche", "gardine", "decke", "decken", "küchenstuhl", "küchentisch",
        "gartenmöbel", "deko"
    ],
    "Werkzeug & Baumarkt": [
        "werkstatt", "akkuschrauber", "bohrer", "schraube",
        "säge", "schweißgerät", "dübel", "mörtel", "leiter",
        "werkzeug", "baumarkt", "hammer", "zange", "elektrowerkzeug", "gartenwerkzeug", "farbe", "pinsel", "spachtel", "schraubenzieher", "sägen", "bohrmaschine",
        "gartengeräte", "rasenmäher", "heckenschere", "gartenschlauch"
    ],
    "Spielzeug & Baby": [
        "lego", "puppe", "spielzeug", "kinderwagen",
        "wickel", "kuscheltier", "baustein", "puzzle", "spiele", "spielware",
        "toys", "games", "wasserspielzeug", "spielware"
    ],
    "Kleidung & Accessoires": [
        "t-shirt", "hose", "jacke", "kleid", "schuh", "sneaker",
        "unterwäsche", "schmuck", "tasche", "gürtel", "mütze",
        "accessoire", "mode", "kleidung", "rock", "jeans", "uhr", "rucksack", "taschen", "körperpflege", "radbekleidung",
        "sportbekleidung", "anzug", "bluse", "sportschuh", "damenschuh", "herrenschuh", "kinderschuh",
        "rucksäcke", "shoes"
    ],
    "Kosmetik & Drogerie": [
        "parfum", "deo", "shampoo", "seife", "make-up", "nagellack", "makeup", "kosmetik", "drogerie", "dusche", "rasur", "spülung", "seife", "zahnpasta",
        "kontaktlinsen", "creme", "lotion", "beauty", "handpflege", "körperpflege", "foundation", "haare",
        "gesichtspflege"
    ],
    "Lebensmittel & Getränke": [
        "kaffee", "spirituose", "tee", "bier", "wein", "snack", "essen", "getränk", "lebensmittel",
        "nahrung", "brot", "whyski", "haushaltsgerät", "kaffee", "tee"
    ],
    "Gesundheit & Pflege": [
        "vitamin", "arzneimittel", "medikament", "vitamin", "verband", "pflaster", "desinfektion", "gesundheit", "pflege", "apotheke", "thermometer", "gesundheit",
        "gesichtspflege", "körperpflege"
    ],
    "Bücher, Filme & Musik": [
        "buch", "film", "dvd", "blu-ray", "cd", "hörbuch", "zeitschrift", "musik", "schallplatte", "hörbuch", "hörspiel", 
    ],
    "Sport & Freizeit": [
        "fahrrad", "helm", "hantel", "yoga", "zelt", "angeln", "ski", "trampolin", "sport", "fitness", "laufen", "joggen", "tennis", "fußball"
    ],
    "Haustier & Tierbedarf": [
        "hundefutter", "katzenstreu", "kratzbaum", "hundespielzeug", "aquarium", "nager", "tier", "haustier", "hund", "katze", "fisch", "vogel", "käfig", "terrarium", "leckerli"
    ],
    "Bürobedarf": [
        "ordner", "hefter", "locher", "druckerpapier", "kugelschreiber", "post-it", "drucker", "scanner", "bürobedarf", "papier", "büro", "stift", "kalender"
    ],
    "Sonstige": [
    ],
    "Undefiniert": [
    ]
}

# Pseudo-Kategorien, die oft als oberste Ebene auftauchen und übersprungen werden sollen
STOPWORDS_FIRST_LEVEL = {
    "damen", "herren", "kinder", "mädchen", "jungen", "baby", "babys",
    "sale", "angebote", "neuheiten", "marke", "brand", "top", "neu"
}

# ------------------------------------------------------------
# 1) HEURISTISCHE ZUORDNUNG
# ------------------------------------------------------------
def normalize_text(s: str) -> str:
    s = s.lower()
    # vereinheitliche Trennzeichen zu ">"
    s = re.sub(r"[|/:\\>]+", ">", s)
    # entferne Sonderzeichen, mehrfachspaces
    s = re.sub(r"[^0-9a-zäöüß><\s\-\.]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s
    
def tokenize(text: str):
    return set(text.lower().split())

def match_with_suffix(word, tokens):
    suffixes = ["", "e", "en", "n", "s", "er"]
    for suf in suffixes:
        if word + suf in tokens:
            return True
    return False
    
def extract_all_meaningful_segments(cat): 
    """ Gibt eine Liste aller Segmente zurück, überspringt dabei Stopwords. """ 
    if not isinstance(cat, str) or not cat.strip(): 
        return None
        
    s = normalize_text(cat) 
        
    parts = [p.strip() for p in s.split(">") if p.strip()] 
    if not parts: 
        return None

    return [p for p in parts if p not in STOPWORDS_FIRST_LEVEL] or [parts[0]]


def heuristic_match(cat):
    """
    Nimmt alle Segmente einer Kategorie, prüft gegen HEURISTIC_RULES.
    Gibt die Kategorie mit den meisten Treffern zurück.
    """
    segments = extract_all_meaningful_segments(cat)
    if segments is None:
        return "Undefiniert"
    else:
        tokens = tokenize(" ".join(segments))
        
        counts = Counter()
        for cat_name, keys in HEURISTIC_RULES.items():
            for k in keys:
                if match_with_suffix(k, tokens):
                    counts[cat_name] += 1
        
        if not counts:
            return "Sonstige"
        
        return counts.most_common(1)[0][0]

# ------------------------------------------------------------
# 2) HAUPTFUNKTION: MAPPING PIPELINE
# ------------------------------------------------------------
def map_shop_categories(df: pd.DataFrame, col: str = "shop_cat") -> pd.DataFrame:
    out = df.copy()
    tqdm.pandas(desc="Prozessiere Kategorien")
    out["top_category_mapped"] = out["shop_cat"].progress_apply(heuristic_match)
    
    return out

# ------------------------------------------------------------
# 3) REPORTING / EXPORT
# ------------------------------------------------------------
def summarize_and_export(out_df: pd.DataFrame, original_col: str = "shop_cat"):
    # Reduktionsübersicht
    n_original = out_df[original_col].nunique(dropna=True)

    print(f"Einzigartige Original-Kategorien: {n_original}")
    print()
    print("Top 20 Mapped Kategorien (Count):")
    print(out_df["top_category_mapped"].value_counts())
    print(out_df.head())

    # Speichere Mapping (ein Datensatz je Zeile, inkl. Score & Quelle)
    mapping = (
        out_df[[original_col,"top_category_mapped"]] # "tfidf_score", ,"heuristic_hit"
        .copy()
    )
    mapping.to_csv("Categories/category_mapping.csv", index=False)

    # Häufigkeiten der Zielkategorien
    counts = out_df["top_category_mapped"].value_counts().rename_axis("top_category").reset_index(name="count")
    counts.to_csv("Categories/top_category_counts.csv", index=False)

    return mapping, counts

# ------------------------------------------------------------
# 4) BEISPIEL-DURCHLAUF (auskommentiert, damit du es selbst steuerst)
# ------------------------------------------------------------
out_df = map_shop_categories(df, col="shop_cat")
mapping, counts = summarize_and_export(out_df, original_col="shop_cat")


# ------------------------------------------------------------
# 5) In pickle umwandeln für spätere Sets
# ------------------------------------------------------------
#out_df.to_pickle("../data/working/dedup_preprocessed_rev2_docs_since_2020_01_01_only_de_strict_only_long_title_only_mainentity_with_new_category.pkl.gz", compression="gzip")

Prozessiere Kategorien: 100%|██████████| 3030931/3030931 [16:37<00:00, 3038.99it/s]


Einzigartige Original-Kategorien: 161947

Top 20 Mapped Kategorien (Count):
top_category_mapped
Kleidung & Accessoires     724985
Sonstige                   576738
Elektronik & Computer      465547
Möbel & Wohnen             347565
Auto & Motorrad            287082
Werkzeug & Baumarkt        161519
Kosmetik & Drogerie        132090
Spielzeug & Baby           124962
Sport & Freizeit            63390
Gesundheit & Pflege         57999
Lebensmittel & Getränke     28338
Undefiniert                 24665
Bürobedarf                  17378
Haustier & Tierbedarf       15998
Bücher, Filme & Musik        2675
Name: count, dtype: int64
             id  product_id  \
0    1817742672  2820098210   
1    4075272753   852480790   
4    2074416376  1217832594   
6    1527509196  1038150077   
8  156275941980  3621538713   

                                                name  \
0  Vipack: Hochbett / Etagenbett "BONNY" Weiß / B...   
1  Sommerreifen PIRELLI P-ZERO (NEW) S.C.  215/45...   
4  Telekom Sp

In [ ]:
print("Amount of distinct shop categories for top category mapped Snstige:", out_df[out_df["top_category_mapped"]=="Sonstige"]["shop_cat"].nunique())

Amount of distinct shop categories for top category mapped Snstige: 65871


In [ ]:
import openai
import os; API_key = os.environ.get('OPENAI_API_KEY')  # set OPENAI_API_KEY env var
import time
import re
from tqdm import tqdm

def normalize(text):
    return re.sub(r"[^a-z0-9]", " ", text.lower())
# Function to call OpenAI API for entity matching
def category_matching(api_key, product, shop_cat, category_list):
    if shop_cat is "Unbekannt":
        prompt = f"Welche shopping categorie würdest du diesem Product: \"{product}\" zuordnen? Du darfst nur diese Kategorien nurtzen: {category_list}."
    else:
        prompt = f"Welche shopping categorie würdest du diesem Product: \"{product}\" mit dieser Shopping Kategorie: \"{shop_cat}\" zuordnen? Du darfst nur diese Kategorien nurtzen: {category_list}."
    openai.api_key = api_key
    #Genauerer befehl -performt schlechter
    #prompt = f"Handelt es sich bei diesen beiden Produkten um dasselbe reale Produkt?\nProdukt 1: {entity_1}\nProdukt 2: {entity_2}\n. Bitte beachte, dass die Farbe, Größe und alle anderen Produkteigenschaften gleich sein müssen, damit man es als gleiches Produkt ansieht. \n.Antworte nur mit Ja oder Nein."
    #Allgemein
    prompt = f"Welche shopping categorie würdest du diesem Product: \"{product}\" mit dieser Shopping Kategorie: \"{shop_cat}\" zuordnen? Du darfst nur diese Kategorien nurtzen: {category_list}."
    
    response = openai.chat.completions.create(
        model="gpt-5-mini",
        messages=[{"role": "user", "content": prompt}]
    )
    usage = response.usage
    input_tokens = usage.prompt_tokens if usage else 0
    output_tokens = usage.completion_tokens if usage else 0

    return response.choices[0].message.content

# Function to safely call the LLM with retries
def safe_category_match(api_key, shop_cat, product, category_list, retries=5, delay=5):
    for attempt in range(1, retries + 1):
        try:
            result = category_matching(api_key, product, shop_cat, category_list)
            # Validate structure
            if result is None and attempt < retries:
                time.sleep(delay)
                continue
            elif result is None and attempt >= retries:
                raise ValueError("Empty result from LLM.")
            
            answer = result.strip().lower()
            # Validate that the answer is one of the expected outputs
            normalized_answer = normalize(answer)
            if any(normalize(category) in normalized_answer for category in category_list):
                return result
            else:
                print(f"[Attempt {attempt}] Invalid answer: '{answer}'")
                if attempt < retries:
                    time.sleep(delay)
                else:
                    raise ValueError("Invalid response from LLM after multiple attempts.")
        except (openai.APIError, openai.RateLimitError, openai.APITimeoutError, openai.APIConnectionError) as e:
            print(f"[Attempt {attempt}] OpenAI API error: {e}")
            if attempt < retries:
                time.sleep(delay)
            else:
                print(" All retries failed. Exiting.")
                raise e  # or handle how you'd like
            
# Run LLM for all items with top mapped category "Sonstige" or "Undefiniert"
try:
    # Filter out only "Sonstige" or "Undefiniert"
    sonstige_df = out_df[
        (out_df["top_category_mapped"] == "Sonstige") |
        (out_df["top_category_mapped"] == "Undefiniert")
    ].copy()

    # Create a dict to store GPT results per shop_cat
    category_cache = {}

    sonstige_df["shop_cat"] = sonstige_df["shop_cat"].fillna("Unbekannt")
    # Loop through each unique shop_cat group
    for shop_cat, group in tqdm(sonstige_df.groupby("shop_cat"), desc="LLM Kategorisierung (per shop_cat)"):
        try:
            # Choose one representative product name (e.g. first one)
            product_example = group["name"].iloc[0]

            # Call LLM once per shop_cat
            mapped_category = safe_category_match(API_key, shop_cat, product_example, TOP_CATEGORIES)
        
            # Store mapping in cache
            category_cache[shop_cat] = mapped_category

            # Apply to all rows in the group
            sonstige_df.loc[group.index, "top_category_mapped"] = mapped_category

        except Exception as e_inner:
            print(f"[Shop-Cat '{shop_cat}'] Error during LLM call: {e_inner}")
            # Keep original mapping if LLM fails
            sonstige_df.loc[group.index, "top_category_mapped"] = group["top_category_mapped"]

    # Update main dataframe
    out_df.loc[sonstige_df.index, "top_category_mapped"] = sonstige_df["top_category_mapped"].values

except Exception as e:
    print(f"\nScript interrupted due to error:\n{e}\n")


In [4]:
import openai
import os; API_key = os.environ.get('OPENAI_API_KEY')  # set OPENAI_API_KEY env var
import time
import re
from tqdm import tqdm

def normalize(text):
    return re.sub(r"[^a-z0-9]", " ", text.lower())
# Function to call OpenAI API for entity matching
def category_matching(api_key, product, shop_cat, category_list):
    openai.api_key = api_key
    #Genauerer befehl -performt schlechter
    #prompt = f"Handelt es sich bei diesen beiden Produkten um dasselbe reale Produkt?\nProdukt 1: {entity_1}\nProdukt 2: {entity_2}\n. Bitte beachte, dass die Farbe, Größe und alle anderen Produkteigenschaften gleich sein müssen, damit man es als gleiches Produkt ansieht. \n.Antworte nur mit Ja oder Nein."
    #Allgemein
    prompt = f"Welche shopping categorie würdest du diesem Product: \"{product}\" mit dieser Shopping Kategorie: \"{shop_cat}\" zuordnen? Du darfst nur diese Kategorien nurtzen: {category_list}."
    
    response = openai.chat.completions.create(
        model="gpt-5-mini",
        messages=[{"role": "user", "content": prompt}]
    )
    usage = response.usage
    input_tokens = usage.prompt_tokens if usage else 0
    output_tokens = usage.completion_tokens if usage else 0

    return response.choices[0].message.content

# Function to safely call the LLM with retries
def safe_category_match(api_key, shop_cat, product, category_list, retries=5, delay=5):
    for attempt in range(1, retries + 1):
        try:
            result = category_matching(api_key, product, shop_cat, category_list)
            # Validate structure
            if result is None and attempt < retries:
                time.sleep(delay)
                continue
            elif result is None and attempt >= retries:
                raise ValueError("Empty result from LLM.")
            
            answer = result.strip().lower()
            # Validate that the answer is one of the expected outputs
            normalized_answer = normalize(answer)
            if any(normalize(category) in normalized_answer for category in category_list):
                return result
            else:
                print(f"[Attempt {attempt}] Invalid answer: '{answer}'")
                if attempt < retries:
                    time.sleep(delay)
                else:
                    raise ValueError("Invalid response from LLM after multiple attempts.")
        except (openai.APIError, openai.RateLimitError, openai.APITimeoutError, openai.APIConnectionError) as e:
            print(f"[Attempt {attempt}] OpenAI API error: {e}")
            if attempt < retries:
                time.sleep(delay)
            else:
                print(" All retries failed. Exiting.")
                raise e  # or handle how you'd like
            
# Run LLM for all items with top mapped category "Sonstige" or "Undefiniert"
try:
    # Filter out only "Sonstige" or "Undefiniert"
    sonstige_df = out_df[
        (out_df["top_category_mapped"] == "Sonstige") |
        (out_df["top_category_mapped"] == "Undefiniert")
    ].copy()

    sonstige_df["shop_cat"] = sonstige_df["shop_cat"].fillna("Unbekannt")

    # Create a dict to store GPT results per shop_cat
    category_cache = {}
    i = 0
    # Loop through each unique shop_cat group
    for shop_cat, group in tqdm(sonstige_df.groupby("shop_cat"), desc="LLM Kategorisierung (per shop_cat)"):
        try:
            # Choose one representative product name (e.g. first one)
            product_example = group["name"].iloc[0]

            # Call LLM once per shop_cat
            mapped_category = f"Test_{i}"

            i = i+1
        
            # Store mapping in cache
            category_cache[shop_cat] = mapped_category

            # Apply to all rows in the group
            sonstige_df.loc[group.index, "top_category_mapped"] = mapped_category

        except Exception as e_inner:
            print(f"[Shop-Cat '{shop_cat}'] Error during LLM call: {e_inner}")
            # Keep original mapping if LLM fails
            sonstige_df.loc[group.index, "top_category_mapped"] = group["top_category_mapped"]

    # Update main dataframe
    out_df.loc[sonstige_df.index, "top_category_mapped"] = sonstige_df["top_category_mapped"].values

    print("Top 20 Mapped Kategorien (Count):")
    print(out_df["top_category_mapped"].value_counts())
    print(out_df.head())

except Exception as e:
    print(f"\nScript interrupted due to error:\n{e}\n")


LLM Kategorisierung (per shop_cat): 100%|██████████| 65878/65878 [00:38<00:00, 1726.20it/s]


Top 20 Mapped Kategorien (Count):
top_category_mapped
Kleidung & Accessoires    724985
Elektronik & Computer     465547
Möbel & Wohnen            347565
Auto & Motorrad           287082
Werkzeug & Baumarkt       161519
                           ...  
Test_37465                     1
Test_54547                     1
Test_50202                     1
Test_49129                     1
Test_5622                      1
Name: count, Length: 65891, dtype: int64
             id  product_id  \
0    1817742672  2820098210   
1    4075272753   852480790   
4    2074416376  1217832594   
6    1527509196  1038150077   
8  156275941980  3621538713   

                                                name  \
0  Vipack: Hochbett / Etagenbett "BONNY" Weiß / B...   
1  Sommerreifen PIRELLI P-ZERO (NEW) S.C.  215/45...   
4  Telekom Speedphone 51 Festnetztelefon (mit Bas...   
6  Spiegelschrank »Basic« 60 cm weiß, Möbelpartne...   
8  Gelenkarmmarkise SPETTMANN STAR Markisen Gr. 3...   

                  

A Manual check of the fisrt 200 categories were done to confirm that the items were corectly placed. All items were sorted correctly

In [4]:
import pandas as pd
df = pd.read_json("../data/derived/gold-standards/products80cc20rnd100un_gs.json.gz", lines=True, compression="gzip")
print(df.shape)
corpus = pd.read_pickle("../data/working/dedup_preprocessed_rev2_docs_since_2020_01_01_only_de_strict_only_long_title_only_mainentity_with_new_category.pkl.gz")

(4500, 15)


In [5]:
# Falls ein product_id mehrfach vorkommt, nimm z. B. den ersten Eintrag
corpus_unique = corpus.drop_duplicates(subset='product_id', keep='first')

# Mapping-Dict erstellen
id_to_cat = corpus_unique.set_index('product_id')['top_category_mapped']

# Kategorien zuordnen
df['top_category_left'] = df['product_id_left'].map(id_to_cat)
df['top_category_right'] = df['product_id_right'].map(id_to_cat)

# Zähle, wie oft jede Kategorie links und rechts vorkommt
left_cat_counts = df['top_category_left'].value_counts().to_dict()
right_cat_counts = df['top_category_right'].value_counts().to_dict()

# Kombiniere beide Counts
combined_cat_counts = {
    cat: left_cat_counts.get(cat, 0) + right_cat_counts.get(cat, 0)
    for cat in set(left_cat_counts) | set(right_cat_counts)
}

# Sortiere absteigend nach Häufigkeit
sorted_combined_cat_counts = dict(sorted(combined_cat_counts.items(), key=lambda x: x[1], reverse=True))

# Ausgabe
for cat, count in sorted_combined_cat_counts.items():
    print(f"Category: {cat}, Count: {count}")


Category: Sonstige, Count: 2433
Category: Möbel & Wohnen, Count: 2403
Category: Kleidung & Accessoires, Count: 2197
Category: Kosmetik & Drogerie, Count: 473
Category: Elektronik & Computer, Count: 428
Category: Werkzeug & Baumarkt, Count: 305
Category: Spielzeug & Baby, Count: 251
Category: Sport & Freizeit, Count: 212
Category: Auto & Motorrad, Count: 161
Category: Gesundheit & Pflege, Count: 95
Category: Bürobedarf, Count: 42
